# CSET485 – AI and Society | Assignment 1
## Adult Census Income Dataset (Roll-number seed: 788)

**Assigned dataset:** Adult Census Income (last roll-number digit = 8)  
**Train/test split:** 70% / 30%, `random_state=788`  
**Target:** income > $50K (1 = yes, 0 = no)

This notebook reproduces the calculations required in the assignment and creates a PDF report with the results generated by this run. It uses both official UCI files (`adult.data` and `adult.test`) to form the 48,842-row Adult dataset. The data are from the 1994 US Census extraction.

**Important:** Run every cell from top to bottom. Do not type or invent numerical results in the report; the tables, charts, and essay evidence are populated from the computed results below.


In [ ]:
# 1. Install/import libraries
%pip -q install pandas numpy scikit-learn statsmodels matplotlib seaborn reportlab

import io, os, re, textwrap, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
from IPython.display import display

warnings.filterwarnings("ignore")
SEED = 788
np.random.seed(SEED)

COLS = [
    "age","workclass","fnlwgt","education","education_num","marital_status",
    "occupation","relationship","race","sex","capital_gain","capital_loss",
    "hours_per_week","native_country","income"
]


In [ ]:
# 2. Load the official UCI Adult files
# The official test file starts with a comment line and has a trailing period in income labels.
TRAIN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

def read_uci_file(url, skiprows=0):
    return pd.read_csv(
        url, header=None, names=COLS, skiprows=skiprows,
        na_values=["?", " ?"], skipinitialspace=True
    )

try:
    adult_train = read_uci_file(TRAIN_URL)
    adult_test = read_uci_file(TEST_URL, skiprows=1)
    df = pd.concat([adult_train, adult_test], ignore_index=True)
    print("Loaded both official UCI files.")
except Exception as e:
    print("Automatic download did not work:", repr(e))
    print("Fallback: upload adult.data and adult.test in the Colab Files panel, then rerun this cell.")
    from google.colab import files
    uploaded = files.upload()
    if "adult.data" not in uploaded or "adult.test" not in uploaded:
        raise ValueError("Please upload both adult.data and adult.test.")
    adult_train = pd.read_csv(io.BytesIO(uploaded["adult.data"]), header=None, names=COLS,
                              na_values=["?"], skipinitialspace=True)
    adult_test = pd.read_csv(io.BytesIO(uploaded["adult.test"]), header=None, names=COLS,
                             skiprows=1, na_values=["?"], skipinitialspace=True)
    df = pd.concat([adult_train, adult_test], ignore_index=True)

# Normalize whitespace and target labels.
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()
df["income"] = df["income"].str.replace(".", "", regex=False).str.strip()
df["income_bin"] = (df["income"] == ">50K").astype(int)

print("Dataset shape:", df.shape)
print("Target counts:")
display(df["income"].value_counts().rename_axis("income").to_frame("count"))
display(df.head())


In [ ]:
# 3. Data quality and preprocessing documentation
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean()*100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_percent": missing_pct})
display(missing_table[missing_table.missing_count > 0])

print("Duplicate rows:", int(df.duplicated().sum()))
print("Numeric summary:")
display(df.describe().T)

# Keep raw income as target only; remove education text because education_num is its ordered numeric encoding.
# fnlwgt is a census sampling weight, not an individual causal attribute; retain it as a feature for the
# required predictive exercise but interpret it cautiously.
TARGET = "income_bin"
DROP = ["income", TARGET, "education"]
X = df.drop(columns=DROP)
y = df[TARGET]

numeric_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"Train rows: {len(X_train):,} ({len(X_train)/len(df):.1%})")
print(f"Test rows:  {len(X_test):,} ({len(X_test)/len(df):.1%})")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

preprocessing_notes = [
    "Missing values: '?' tokens were read as missing. Numeric columns are median-imputed; categorical columns are imputed with their most frequent training category inside the pipeline.",
    "Categorical encoding: one-hot encoding with handle_unknown='ignore'; the encoder is fitted only on training data.",
    "Feature scaling: numeric predictors are standardized using training-set means and standard deviations inside the pipeline.",
    "Outliers: no rows are deleted solely for extreme numeric values. Capital gain/loss are highly skewed and retained; their influence is considered in the limitations. This avoids silently removing potentially meaningful high-income cases.",
    "Leakage prevention: train/test split is stratified and uses random_state=788; imputation, encoding, and scaling are fitted only on the training partition."
]
for note in preprocessing_notes: print("•", note)


In [ ]:
# 4. Part 1 — Model A (OLS Linear Regression) and Model B (Logistic Regression)
ols = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression())
])
logit = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000, solver="liblinear", random_state=SEED))
])

ols.fit(X_train, y_train)
logit.fit(X_train, y_train)

ols_pred = ols.predict(X_test)                  # continuous OLS score
logit_prob = logit.predict_proba(X_test)[:, 1]  # probability for income > $50K

median_ols = float(np.median(ols_pred))
ols_class = (ols_pred >= median_ols).astype(int)
logit_class = (logit_prob >= 0.5).astype(int)

agreement_cm = confusion_matrix(ols_class, logit_class, labels=[0,1])
mismatch = int(np.sum(ols_class != logit_class))
mismatch_pct = mismatch / len(y_test) * 100

print(f"Median OLS predicted value (threshold): {median_ols:.6f}")
print("Agreement confusion matrix (rows = OLS threshold class, columns = Logistic class):")
display(pd.DataFrame(agreement_cm,
    index=["OLS low (0)","OLS high (1)"],
    columns=["Logistic low (0)","Logistic high (1)"]))
print(f"Mismatched classifications: {mismatch:,} / {len(y_test):,} = {mismatch_pct:.2f}%")

model_metrics = pd.DataFrame({
    "Metric": ["Accuracy","Precision","Recall","F1","ROC-AUC"],
    "Logistic model": [
        accuracy_score(y_test, logit_class),
        precision_score(y_test, logit_class, zero_division=0),
        recall_score(y_test, logit_class, zero_division=0),
        f1_score(y_test, logit_class, zero_division=0),
        roc_auc_score(y_test, logit_prob)
    ]
})
display(model_metrics.round(4))


In [ ]:
# 5. Part 2.1 — Simpson's Paradox Hunt (subgroup analysis by sex)
# A transparent, interpretable statsmodels GLM is used here so coefficients and confidence intervals
# can be reported. The key predictor is hours_per_week; controls include age, education, capital gain/loss,
# marital status and race. The sex variable is omitted inside sex-specific models.
formula_full = "income_bin ~ hours_per_week + age + education_num + capital_gain + capital_loss + C(marital_status) + C(race) + C(sex)"
formula_sub = "income_bin ~ hours_per_week + age + education_num + capital_gain + capital_loss + C(marital_status) + C(race)"

def fit_glm(formula, data):
    return smf.glm(formula=formula, data=data, family=sm.families.Binomial()).fit()

analysis_df = df.copy()
simpson_models = {
    "Full dataset": fit_glm(formula_full, analysis_df),
    "Female subgroup": fit_glm(formula_sub, analysis_df[analysis_df.sex == "Female"]),
    "Male subgroup": fit_glm(formula_sub, analysis_df[analysis_df.sex == "Male"])
}
simpson_rows = []
for label, model in simpson_models.items():
    ci = model.conf_int().loc["hours_per_week"]
    simpson_rows.append({
        "Dataset": label,
        "hours_per_week coefficient": model.params["hours_per_week"],
        "95% CI lower": ci.iloc[0],
        "95% CI upper": ci.iloc[1],
        "p-value": model.pvalues["hours_per_week"]
    })
simpson_table = pd.DataFrame(simpson_rows).set_index("Dataset")
display(simpson_table.round(6))
signs = np.sign(simpson_table["hours_per_week coefficient"].values)
if len(set(signs)) > 1:
    simpson_finding = "A coefficient sign reversal is present across the full and sex-specific models."
else:
    simpson_finding = "No sign reversal was observed for hours_per_week; compare coefficient magnitudes and confidence intervals for subgroup effect differences."
print(simpson_finding)


In [ ]:
# 6. Part 2.2 — Omitted Variable Bias
# Deliberately omit education_num, a plausible confounder of income and work-hours/age associations.
full_ovb_formula = "income_bin ~ age + education_num + hours_per_week + capital_gain + capital_loss + C(marital_status) + C(sex) + C(race)"
reduced_ovb_formula = "income_bin ~ age + hours_per_week + capital_gain + capital_loss + C(marital_status) + C(sex) + C(race)"
full_ovb = fit_glm(full_ovb_formula, analysis_df)
reduced_ovb = fit_glm(reduced_ovb_formula, analysis_df)

predictors_compare = ["age","hours_per_week","capital_gain","capital_loss"]
ovb_rows = []
for p in predictors_compare:
    b1, b2 = full_ovb.params[p], reduced_ovb.params[p]
    pct = np.nan if abs(b1) < 1e-12 else (b2-b1)/abs(b1)*100
    ovb_rows.append({"Predictor":p, "Full coefficient":b1, "Reduced coefficient":b2,
                     "Absolute percentage change (%)":abs(pct), "Signed change (%)":pct})
ovb_table = pd.DataFrame(ovb_rows).set_index("Predictor")
display(ovb_table.round(5))
largest_ovb = ovb_table["Absolute percentage change (%)"].idxmax()
print("Largest coefficient shift among the four reported predictors:", largest_ovb)


In [ ]:
# 7. Part 2.3 — Adversarial subset: confident but incorrect predictions
test_results = X_test.copy()
test_results["actual"] = y_test.to_numpy()
test_results["probability"] = logit_prob
test_results["predicted"] = logit_class
test_results["error_type"] = np.where(
    (test_results.probability >= .75) & (test_results.actual == 0), "False positive",
    np.where((test_results.probability <= .25) & (test_results.actual == 1), "False negative", "Other")
)
wrong_confident = test_results[test_results.error_type != "Other"].copy()
n_confident_wrong = len(wrong_confident)
pct_confident_wrong = n_confident_wrong / len(test_results) * 100

if len(wrong_confident) > 50:
    adversarial = wrong_confident.sample(n=50, random_state=SEED)
else:
    adversarial = wrong_confident.copy()

print(f"High-confidence wrong predictions: {n_confident_wrong:,} ({pct_confident_wrong:.2f}% of test set)")
print(f"Rows selected for adversarial analysis: {len(adversarial):,}")
if len(adversarial):
    print("Age distribution — adversarial subset:")
    display(adversarial.age.describe().to_frame().T)
    print("Sex breakdown — adversarial subset (%):")
    display((adversarial.sex.value_counts(normalize=True)*100).round(2).to_frame("percent"))
    print("Sex breakdown — full test set (%):")
    display((test_results.sex.value_counts(normalize=True)*100).round(2).to_frame("percent"))
    print("Most frequent categorical characteristics in adversarial subset:")
    for c in ["workclass","occupation","marital_status","race","relationship","native_country"]:
        display((adversarial[c].value_counts(normalize=True).head(5)*100).round(1).to_frame("percent").rename_axis(c))
    common_col = "marital_status"
    common_value = adversarial[common_col].mode().iloc[0]
    common_pct = (adversarial[common_col] == common_value).mean()*100
    print(f"Most common {common_col}: {common_value} ({common_pct:.1f}% of selected subset)")
else:
    common_value, common_pct = "No qualifying cases", 0.0
    print("No cases met the specified confidence/error criteria. Report this result; do not manufacture a 50-row subset.")


In [ ]:
# 8. Part 2.4 — Fairness trade-off across race groups (White vs Black)
# Assignment asks for a two-group comparison. Other race categories are excluded from this specific
# pairwise metric calculation and this limitation is stated in the report.
fair = test_results[test_results.race.isin(["White","Black"])].copy()
fairness_rows = []

def rates_for_group(g, threshold):
    y_true = g.actual.to_numpy()
    y_hat = (g.probability.to_numpy() >= threshold).astype(int)
    positives = y_hat == 1
    negatives = y_hat == 0
    actual_pos = y_true == 1
    actual_neg = y_true == 0
    dp = positives.mean() if len(g) else np.nan
    tpr = ((y_hat == 1) & actual_pos).sum()/actual_pos.sum() if actual_pos.sum() else np.nan
    fpr = ((y_hat == 1) & actual_neg).sum()/actual_neg.sum() if actual_neg.sum() else np.nan
    pp = y_true[positives].mean() if positives.sum() else np.nan
    return dp, tpr, fpr, pp

def ratio(a,b):
    if pd.isna(a) or pd.isna(b): return np.nan
    if b == 0: return np.nan
    return a/b

for t in np.round(np.arange(.05, .951, .10), 2):
    white = rates_for_group(fair[fair.race=="White"], t)
    black = rates_for_group(fair[fair.race=="Black"], t)
    dp_ratio = ratio(white[0], black[0])
    tpr_ratio = ratio(white[1], black[1])
    fpr_ratio = ratio(white[2], black[2])
    eo = min(tpr_ratio, fpr_ratio) if not (pd.isna(tpr_ratio) or pd.isna(fpr_ratio)) else np.nan
    pp_ratio = ratio(white[3], black[3])
    fairness_rows.append({"Threshold":t, "DP ratio (White/Black)":dp_ratio,
                          "EO ratio":eo, "PP ratio (White/Black)":pp_ratio,
                          "White selection rate":white[0], "Black selection rate":black[0],
                          "White TPR":white[1], "Black TPR":black[1],
                          "White FPR":white[2], "Black FPR":black[2]})
fairness_table = pd.DataFrame(fairness_rows)
display(fairness_table.round(4))

plt.figure(figsize=(8,6))
plt.scatter(fairness_table["DP ratio (White/Black)"], fairness_table["EO ratio"])
for _, r in fairness_table.iterrows():
    if pd.notna(r["DP ratio (White/Black)"]) and pd.notna(r["EO ratio"]):
        plt.annotate(f'{r["Threshold"]:.2f}', (r["DP ratio (White/Black)"], r["EO ratio"]),
                     xytext=(5,5), textcoords="offset points")
plt.xlim(0,1.5); plt.ylim(0,1.5)
plt.xlabel("Demographic Parity ratio (White / Black)")
plt.ylabel("Equalized Odds ratio (min TPR ratio, FPR ratio)")
plt.title("Fairness trade-off across decision thresholds")
plt.grid(alpha=.25); plt.tight_layout()
tradeoff_chart = "/content/fairness_tradeoff.png"
plt.savefig(tradeoff_chart, dpi=160)
plt.show()

# Report predictive parity as a third metric. Ratios near 1 indicate parity for the selected pair,
# but ratios alone can obscure small denominators and do not establish ethical acceptability.


In [ ]:
# 9. Choose a defensible threshold using transparent criteria
# Illustrative selection rule: among thresholds with finite DP/EO/PP ratios, choose the one minimizing
# total absolute distance from 1. This is a descriptive compromise, NOT a universally correct ethical rule.
valid = fairness_table.dropna(subset=["DP ratio (White/Black)","EO ratio","PP ratio"]).copy()
if len(valid):
    valid["parity_distance"] = (
        (valid["DP ratio (White/Black)"]-1).abs() +
        (valid["EO ratio"]-1).abs() +
        (valid["PP ratio (White/Black)"]-1).abs()
    )
    chosen_row = valid.loc[valid.parity_distance.idxmin()]
    chosen_threshold = float(chosen_row["Threshold"])
    print("Illustrative compromise threshold:", chosen_threshold)
    display(chosen_row.to_frame("value"))
else:
    chosen_threshold = 0.5
    chosen_row = fairness_table.iloc[(fairness_table["Threshold"]-0.5).abs().argmin()]
    print("No threshold has all three finite ratios; using 0.50 only as a reference and documenting undefined ratios.")


## 10. Part 3 and Part 4 — Automatically generated evidence-based report

The next cell assembles a PDF from the computed tables, plots, and written interpretations. It includes all eight assignment questions and a reflection. Review the generated prose before submission, especially the policy recommendations: fairness metrics are descriptive evidence, not a substitute for stakeholder consultation or legal review.


In [ ]:
# 10. Generate a submission-ready PDF using the actual outputs from this run
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

def fmt(x, digits=4):
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return "NA"

def table_data(df_in, max_rows=30):
    d = df_in.copy().head(max_rows)
    return [[str(c) for c in [d.index.name or ""] + list(d.columns)]] + [
        [str(idx)] + [fmt(v,4) if isinstance(v,(float,np.floating)) else str(v) for v in row]
        for idx,row in zip(d.index, d.to_numpy())
    ]

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="SmallBody", parent=styles["BodyText"], fontSize=8.5, leading=11, spaceAfter=6))
styles.add(ParagraphStyle(name="QHeading", parent=styles["Heading2"], fontSize=11, leading=14, spaceBefore=10, spaceAfter=5))
story = []
story.append(Paragraph("CSET485 – AI and Society | Assignment 1", styles["Title"]))
story.append(Paragraph("Adult Census Income Dataset | Random seed = 788", styles["Heading2"]))
story.append(Paragraph("Student name: ____________________ &nbsp;&nbsp; Roll number: ____________________", styles["BodyText"]))
story.append(Spacer(1,10))
story.append(Paragraph("<b>Dataset and task.</b> The Adult Census Income dataset contains census-derived demographic and employment attributes. The target is whether annual income exceeds $50,000 (1 = yes, 0 = no). The two official UCI files were combined; the resulting dataset has %s rows and %s columns. The 70:30 stratified split used random_state=788." % (f"{len(df):,}", len(df.columns)), styles["SmallBody"]))
story.append(Paragraph("<b>Preprocessing.</b> " + " ".join(preprocessing_notes), styles["SmallBody"]))
story.append(Paragraph("Part 1 — Build and compare models", styles["Heading1"]))
story.append(Paragraph(f"OLS median prediction threshold = {median_ols:.6f}. Logistic probability threshold = 0.50. The agreement confusion matrix is arranged with OLS classes as rows (low, high) and logistic classes as columns (low, high). Mismatches = {mismatch:,} of {len(y_test):,} test cases ({mismatch_pct:.2f}%).", styles["SmallBody"]))
cm_data = [["", "Logistic low", "Logistic high"], ["OLS low", str(agreement_cm[0,0]), str(agreement_cm[0,1])], ["OLS high", str(agreement_cm[1,0]), str(agreement_cm[1,1])]]
t=Table(cm_data, hAlign="LEFT"); t.setStyle(TableStyle([("GRID",(0,0),(-1,-1),.4,colors.grey),("BACKGROUND",(0,0),(-1,0),colors.lightgrey)])); story.append(t)
story.append(Spacer(1,8))
story.append(Paragraph("Logistic test metrics: " + "; ".join(f"{r['Metric']} = {r['Logistic model']:.4f}" for _,r in model_metrics.iterrows()), styles["SmallBody"]))
story.append(Paragraph("<b>Interpretation.</b> OLS is trained to minimize squared error and can produce values outside [0,1]; its median prediction threshold divides the test predictions by their own distribution. Logistic regression estimates class probabilities and uses a fixed 0.50 cutoff. These distinct scales, loss functions, and cutoffs explain why they need not flag the same people.", styles["SmallBody"]))

story.append(Paragraph("Part 2 — The Trap", styles["Heading1"]))
story.append(Paragraph("2.1 Simpson's Paradox Hunt — subgroup analysis by sex", styles["Heading2"]))
story.append(Paragraph("Key predictor: hours_per_week. The following coefficients are log-odds changes per additional weekly work hour, conditional on the listed controls.", styles["SmallBody"]))
story.append(Table(table_data(simpson_table), repeatRows=1, hAlign="LEFT", style=TableStyle([("GRID",(0,0),(-1,-1),.35,colors.grey),("BACKGROUND",(0,0),(-1,0),colors.lightgrey),("FONTSIZE",(0,0),(-1,-1),7)])))
story.append(Paragraph(simpson_finding + " " + ("The sign pattern is " + ", ".join(f"{x}: {v:+.5f}" for x,v in simpson_table["hours_per_week coefficient"].items()) + ".") , styles["SmallBody"]))
story.append(Paragraph("Policy implication: a pooled coefficient should not be treated as proof that the predictor has the same association for every sex group. Decision-makers should inspect subgroup confidence intervals, sample sizes, and error rates before adopting a uniform intervention.", styles["SmallBody"]))

story.append(Paragraph("2.2 Omitted-variable bias", styles["Heading2"]))
story.append(Paragraph("Education_num was deliberately removed from the reduced model because education is plausibly associated with both employment patterns and income. Percentage change is calculated as |reduced − full| / |full| × 100.", styles["SmallBody"]))
story.append(Table(table_data(ovb_table), repeatRows=1, hAlign="LEFT", style=TableStyle([("GRID",(0,0),(-1,-1),.35,colors.grey),("BACKGROUND",(0,0),(-1,0),colors.lightgrey),("FONTSIZE",(0,0),(-1,-1),7)])))
story.append(Paragraph(f"The largest absolute percentage shift among the four reported predictors is for {largest_ovb}. This demonstrates that a coefficient can absorb part of the association carried by an omitted confounder; a model that omits relevant variables can therefore give misleading explanations even when it predicts reasonably well.", styles["SmallBody"]))
story.append(Paragraph("2.3 Adversarial subset construction", styles["Heading2"]))
story.append(Paragraph(f"The test set contains {n_confident_wrong:,} confident errors ({pct_confident_wrong:.2f}% of the test set), defined exactly as P≥0.75 with actual 0 or P≤0.25 with actual 1. The selected subset contains {len(adversarial)} rows (all qualifying rows if fewer than 50, otherwise a reproducible random sample of 50).", styles["SmallBody"]))
if len(adversarial):
    story.append(Paragraph(f"Within the selected subset, the most frequent marital-status category is {common_value}, appearing in {common_pct:.1f}% of selected cases. The age mean is {adversarial.age.mean():.2f} years (median {adversarial.age.median():.1f}); female share is {(adversarial.sex=='Female').mean()*100:.1f}% and male share is {(adversarial.sex=='Male').mean()*100:.1f}%. Compare these with the full test-set distributions shown in the notebook.", styles["SmallBody"]))
else:
    story.append(Paragraph("No qualifying confident errors were found; consequently, no artificial 50-row subset is reported.", styles["SmallBody"]))
story.append(Paragraph("Confident errors can occur when the model's additive logit specification misses interactions or nonlinearities, when rare combinations are poorly represented, or when the observed features do not capture relevant circumstances. A high predicted probability is not a guarantee of correctness.", styles["SmallBody"]))

story.append(Paragraph("2.4 Fairness trade-off analysis", styles["Heading2"]))
story.append(Paragraph("The pairwise comparison uses White and Black test-set records; other race categories are excluded from this two-group calculation. DP = White selection rate / Black selection rate. EO = min(TPR_White/TPR_Black, FPR_White/FPR_Black). PP = precision_White / precision_Black. Values near 1 indicate parity for the corresponding ratio; undefined values arise where a denominator group rate is zero.", styles["SmallBody"]))
story.append(Table(table_data(fairness_table.set_index("Threshold")), repeatRows=1, hAlign="LEFT", style=TableStyle([("GRID",(0,0),(-1,-1),.3,colors.grey),("BACKGROUND",(0,0),(-1,0),colors.lightgrey),("FONTSIZE",(0,0),(-1,-1),6.5)])))
if os.path.exists(tradeoff_chart):
    story.append(RLImage(tradeoff_chart, width=5.6*inch, height=4.2*inch))
story.append(Paragraph(f"An illustrative compromise rule that minimizes the sum of absolute distances of DP, EO and PP ratios from 1 selects threshold {chosen_threshold:.2f}. This is a transparent numerical compromise, not a uniquely correct ethical threshold. It can still sacrifice one metric and should be reviewed with affected communities and domain experts.", styles["SmallBody"]))

story.append(PageBreak())
story.append(Paragraph("Part 3 — Descriptive analysis questions", styles["Heading1"]))

q1 = f"""The OLS model predicts a continuous value and is converted into a class using the median of its test predictions, {median_ols:.6f}. This threshold is data-dependent: approximately half of the test predictions lie on either side, subject to ties. Logistic regression instead estimates the probability of income above $50K and uses the fixed threshold 0.50. These cutoffs answer different questions. The OLS cutoff identifies people above the median OLS score, whereas the logistic cutoff identifies people whose estimated probability of the positive class is at least one half. The agreement matrix is [[{agreement_cm[0,0]}, {agreement_cm[0,1]}], [{agreement_cm[1,0]}, {agreement_cm[1,1]}]], with OLS classes on rows and logistic classes on columns. Thus {mismatch:,} of {len(y_test):,} people ({mismatch_pct:.2f}%) receive different labels. The mismatch is not necessarily a coding error: OLS minimizes squared prediction error and is not constrained to probabilities between zero and one; logistic regression uses a likelihood objective and a probability scale. A median threshold also depends on the distribution of OLS predictions, while 0.50 has a probability interpretation. The two methods can rank individuals differently because their fitted functional forms and loss functions differ. For a real income-related decision, the mismatch matters because two otherwise identical policy pipelines could treat the same person differently. The threshold should therefore be selected and justified in relation to the decision's consequences, not chosen merely because it is conventional."""
q2 = f"""I divided the data by sex and estimated a binomial GLM in the full dataset, the female subgroup and the male subgroup. The key predictor was hours_per_week, with age, education_num, capital_gain, capital_loss, marital status and race included as controls. The estimated hours_per_week coefficients were {simpson_table.iloc[0,0]:.6f} in the full dataset, {simpson_table.iloc[1,0]:.6f} among women and {simpson_table.iloc[2,0]:.6f} among men. Their 95% confidence intervals were [{simpson_table.iloc[0,1]:.6f}, {simpson_table.iloc[0,2]:.6f}], [{simpson_table.iloc[1,1]:.6f}, {simpson_table.iloc[1,2]:.6f}] and [{simpson_table.iloc[2,1]:.6f}, {simpson_table.iloc[2,2]:.6f}], respectively. {simpson_finding} These coefficients describe conditional associations in this historical dataset; they do not establish that working additional hours causes a particular income outcome. A pooled estimate can conceal subgroup differences because the groups may have different distributions of occupation, family status, education or working hours. If the confidence intervals overlap substantially, the evidence for a subgroup difference is weaker; if they are clearly separated, a single pooled effect is especially misleading. A policymaker who ignores subgroup results might assume that a uniform hours-based rule has the same implications for everyone. A more responsible policy would disclose subgroup performance and investigate the mechanisms behind any differences before using the model to allocate opportunities or benefits."""
q3 = f"""For the omitted-variable test, I removed education_num from the full logistic regression. Education is a plausible confounder because it is associated with income and can also be related to age, work hours and employment circumstances. The coefficient comparison is calculated on the same observations and with the same outcome. For age, the full coefficient was {ovb_table.loc['age','Full coefficient']:.6f} and the reduced coefficient was {ovb_table.loc['age','Reduced coefficient']:.6f}, an absolute percentage change of {ovb_table.loc['age','Absolute percentage change (%)']:.2f}%. For hours_per_week, the values were {ovb_table.loc['hours_per_week','Full coefficient']:.6f} and {ovb_table.loc['hours_per_week','Reduced coefficient']:.6f}, a change of {ovb_table.loc['hours_per_week','Absolute percentage change (%)']:.2f}%. For capital_gain, they were {ovb_table.loc['capital_gain','Full coefficient']:.6f} and {ovb_table.loc['capital_gain','Reduced coefficient']:.6f}, a change of {ovb_table.loc['capital_gain','Absolute percentage change (%)']:.2f}%. For capital_loss, they were {ovb_table.loc['capital_loss','Full coefficient']:.6f} and {ovb_table.loc['capital_loss','Reduced coefficient']:.6f}, a change of {ovb_table.loc['capital_loss','Absolute percentage change (%)']:.2f}%. The largest shift among these predictors was {largest_ovb}. This demonstrates that the reduced model reallocates some association previously represented by education to the remaining predictors. The exact shift depends on the correlation structure and model specification. A model that has not been stress-tested may therefore produce coefficients that look precise but partly reflect omitted information. Practitioners should use domain knowledge, sensitivity analyses and external validation rather than interpreting every fitted coefficient as an isolated effect."""
q4 = """A practitioner cannot prove that no hidden confound exists merely by inspecting the fitted model. I would establish a recurring independent validation and audit process. First, domain experts who understand the decision context should review the feature list and data-generation process to identify plausible omitted causes, proxy variables and selection mechanisms. For an income model, this could include local labour-market conditions, job seniority, caregiving constraints, regional wage differences or historical access to education. Second, I would validate the model on a later time period and, where legally and ethically appropriate, on a different geographic sample. A sharp drop in calibration, accuracy or subgroup performance would indicate that the training relationships do not transport reliably. Third, I would inspect residuals and calibration curves by relevant groups and across ranges of age, education and hours worked. Systematic patterns suggest model misspecification or missing interactions. Fourth, I would run sensitivity analyses: add or remove plausible variables, compare coefficient changes, and test interaction terms specified before examining outcomes. Where a credible instrument and defensible assumptions exist, an instrumental-variable design may help investigate causal questions, although it should not be used casually. Finally, I would document data provenance, limitations, model versions and monitoring triggers. This combination cannot reveal every unobserved variable, but it makes uncertainty visible and provides a concrete process for detecting instability before the model is used in a high-stakes setting."""
q5 = f"""The adversarial subset was defined before inspecting individual rows: a false positive is a test case with predicted probability at least 0.75 but actual income class 0; a false negative has predicted probability at most 0.25 but actual class 1. The model produced {n_confident_wrong:,} such confident errors, equal to {pct_confident_wrong:.2f}% of the full test set. The reproducible selected subset contains {len(adversarial)} cases. The most common marital-status category in that subset was {common_value}, representing {common_pct:.1f}% of the selected rows. Its mean age was {adversarial.age.mean():.2f} years, compared with {test_results.age.mean():.2f} years in the full test set. The female share in the subset was {(adversarial.sex=='Female').mean()*100:.1f}%, compared with {(test_results.sex=='Female').mean()*100:.1f}% in the full test set. These comparisons help identify whether the errors cluster in a demographic or social profile; the full category tables in the notebook should also be examined before claiming that one characteristic explains the errors. A logistic model can be confidently wrong because its probability is conditional on the variables and functional form supplied to it. If an important interaction is absent, if a category is rare, or if the relationship is nonlinear, the model may assign a strong probability based on an incomplete pattern. Confidence reflects the fitted model, not certainty about an individual. This subset therefore provides a practical audit list for investigating systematic failure modes."""
q6 = f"""The fairness trade-off chart compares White and Black test-set groups across thresholds. At the selected illustrative threshold {chosen_threshold:.2f}, the DP ratio is {chosen_row['DP ratio (White/Black)']:.4f}, the EO ratio is {chosen_row['EO ratio']:.4f}, and the PP ratio is {chosen_row['PP ratio (White/Black)']:.4f} (when defined). Consider two applicants with the same true income outcome but from different groups. If the threshold is changed to increase the White-to-Black selection-rate ratio toward one, some people in the group that previously had a higher selection rate may lose positive classifications, while more people in the other group may be selected. This may improve demographic parity but alter true-positive and false-positive rates, thereby worsening equalized odds. Conversely, a threshold that improves error-rate balance may leave selection rates unequal. The concrete harm is not abstract: a false negative could deny a person access to a programme or opportunity, while a false positive could direct scarce support away from someone else. The exact individual classifications depend on their predicted probabilities; the group ratios alone do not identify particular people. The notebook provides the threshold-specific selection and error rates needed to illustrate the size and direction of the differences. Because the metrics can conflict, the decision-maker must state which harms are most serious in the actual policy context and explain why."""
q7 = f"""I would consider threshold {chosen_threshold:.2f} as an illustrative compromise, not as a mathematically or morally optimal answer. At this threshold, the White-to-Black demographic-parity ratio is {chosen_row['DP ratio (White/Black)']:.4f}, the equalized-odds ratio is {chosen_row['EO ratio']:.4f}, and the predictive-parity ratio is {chosen_row['PP ratio (White/Black)']:.4f}. The three values should be considered together rather than selecting the threshold that makes only one metric look best. The model's overall test accuracy at its original 0.50 logistic cutoff is {accuracy_score(y_test,logit_class):.4f}; this is useful context but does not guarantee equitable error rates. A defensible threshold must also be evaluated against the practical cost of false positives and false negatives, the size of each group, and uncertainty in the estimated rates. A critic could argue that minimizing the sum of distances from one gives equal numerical weight to three metrics that may have very different social consequences. That criticism is valid: the selection rule is a transparent descriptive heuristic, not a normative principle. In deployment, affected stakeholders, legal requirements and domain experts should help determine which errors matter most. I would therefore use this threshold only for further evaluation, publish the full sweep, and require monitoring rather than treating the number as a final approval."""
q8 = f"""For this income-classification context, I would not prioritize demographic parity or equalized odds in isolation. If the model were used to allocate a scarce opportunity, I would first examine equalized odds because unequal false-positive and false-negative rates mean that people with the same actual outcome may face different error risks across groups. However, if the system is intended to distribute outreach or access, unequal selection rates captured by demographic parity may also be directly consequential. In this analysis, the chosen illustrative threshold {chosen_threshold:.2f} produces a DP ratio of {chosen_row['DP ratio (White/Black)']:.4f} and an EO ratio of {chosen_row['EO ratio']:.4f}; its PP ratio is {chosen_row['PP ratio (White/Black)']:.4f}. These empirical results show how the metrics differ for the selected two-group comparison. Prioritizing equalized odds may sacrifice demographic parity, meaning groups can still receive positive classifications at different rates even when their conditional error rates are more balanced. I would accept that trade-off only if the decision context genuinely makes error-rate equality the more relevant protection, the residual selection-rate gap is disclosed, and affected groups are consulted. If the system is a gatekeeper for essential resources, the safer recommendation may be not to deploy until the institution can justify its threshold and demonstrate that the model improves on the existing process. Fairness is a contextual policy choice, not a property that can be settled by a single ratio."""
questions = [q1,q2,q3,q4,q5,q6,q7,q8]
for i, q in enumerate(questions, 1):
    story.append(Paragraph(f"Question {i}", styles["QHeading"]))
    story.append(Paragraph(q, styles["SmallBody"]))

story.append(PageBreak())
story.append(Paragraph("Part 4 — Reflection", styles["Heading1"]))
reflection = f"""I would recommend <b>conditional non-deployment at this stage</b>: the model may be useful as an audited research prototype, but it should not independently determine access to income-related opportunities or services. The reason is that its overall predictive performance does not settle whether its errors and consequences are acceptable across groups.

A single number fails for at least three reasons. First, an overall score averages over people and can hide subgroup differences. The analysis therefore estimates separate hours-per-week coefficients for women and men, alongside the pooled estimate. The coefficients are {simpson_table.iloc[0,0]:.5f} overall, {simpson_table.iloc[1,0]:.5f} for women and {simpson_table.iloc[2,0]:.5f} for men. Their confidence intervals and p-values should be considered before assuming that the association is identical across groups. Second, a single accuracy value depends on a threshold and does not reveal the relative costs of false positives and false negatives. The logistic model's accuracy at 0.50 is {accuracy_score(y_test,logit_class):.4f}, while the OLS-median and logistic-0.50 classifications disagree for {mismatch:,} people ({mismatch_pct:.2f}%). The same model can therefore produce different policy outcomes simply by changing the decision rule.

Third, fairness measures can conflict. At threshold {chosen_threshold:.2f}, the White-to-Black DP ratio is {chosen_row['DP ratio (White/Black)']:.4f}, EO is {chosen_row['EO ratio']:.4f}, and PP is {chosen_row['PP ratio (White/Black)']:.4f}. A threshold that improves one ratio may worsen another. Fourth, confident model errors matter even when average performance looks acceptable: {n_confident_wrong:,} test cases ({pct_confident_wrong:.2f}%) met the predefined high-confidence error criteria. Fifth, the omitted-variable experiment shows that dropping education_num changes the remaining coefficients; the largest relative shift among the four examined predictors is for {largest_ovb}. A model cannot establish that unobserved confounding is absent simply because its output is precise.

The appropriate next step is a limited, monitored evaluation with human review, subgroup reporting, external validation and a clear appeal process. The model should not be used as the sole basis for consequential decisions. Responsible AI deployment requires transparent assumptions, measured trade-offs, and accountability to affected people—not one headline metric."""
story.append(Paragraph(reflection, styles["SmallBody"]))
story.append(Spacer(1,10))
story.append(Paragraph("<b>Limitations.</b> The Adult data are derived from the 1994 US Census and should not be assumed to represent current labour markets or other countries. Race is simplified to a White-versus-Black comparison for the specified two-group fairness calculations; other groups are not represented in those ratios. Observational associations are not causal effects. Ratios may be undefined when a group-specific denominator rate is zero. This report is a reproducible academic analysis, not a validated decision system.", styles["SmallBody"]))
story.append(Paragraph("<b>Source:</b> Becker, B. & Kohavi, R. (1996). Adult. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20", styles["SmallBody"]))

pdf_path = "/content/CSET485_Adult_Income_Assignment_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=42, leftMargin=42, topMargin=42, bottomMargin=42)
doc.build(story)
print("PDF report created:", pdf_path)
print("Download it from the Colab file panel after running this cell.")


## Submission checklist
- [ ] Run all cells from top to bottom in Google Colab.
- [ ] Inspect the printed tables and the generated PDF; verify the coefficients, percentages, and fairness ratios.
- [ ] Add your name and full roll number to the PDF.
- [ ] Include the notebook and PDF in your submission.
- [ ] Upload the notebook to GitHub and submit its repository link as required by the assignment.
- [ ] Do not claim a Simpson sign reversal unless the computed coefficients actually reverse sign. The notebook explicitly reports if only subgroup effect-strength differences are found.
